In [ ]:
!pip install langchain-huggingface -q

In [ ]:
# ─────────────────────────────────────────────
# Part 1. 라이브러리 설치 및 드라이브 마운트
# ─────────────────────────────────────────────
!pip install sentence-transformers langchain-huggingface langchain-community faiss-gpu-cu12

import json
import numpy as np
import os
import shutil
from google.colab import drive
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

drive.mount('/content/drive')
print("라이브러리 임포트 및 드라이브 마운트 완료")

# ─────────────────────────────────────────────
# Part 2. 설정값 (여기만 바꾸면 됨)
# ─────────────────────────────────────────────
PRESET_ID      = 1                          # ← 1, 2, 3, 4 중 선택
FILE_PATH      = f"/content/ingredient_chunks_preset{PRESET_ID}.json"
FAISS_SAVE_PATH = f"/content/drive/MyDrive/data/faiss_index_preset{PRESET_ID}"
BATCH_SIZE     = 2000                       # 배치 단위 (메모리 안정성)

# ─────────────────────────────────────────────
# Part 3. 청크 파일 로드 + 검증
# ─────────────────────────────────────────────
with open(FILE_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"총 청크 수: {len(all_chunks):,}개")
assert len(all_chunks) > 40000, f"❌ 청크 수 이상함: {len(all_chunks)}"
print("✅ 청크 수 정상")

# LangChain Document 변환
docs = [
    Document(
        page_content=chunk["page_content"],
        metadata=chunk["metadata"]
    )
    for chunk in all_chunks
]
print(f"Document 변환 완료: {len(docs):,}개")

# ─────────────────────────────────────────────
# Part 4. 임베딩 모델 로드
# ─────────────────────────────────────────────
embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
test_vector = embedding_model.embed_query("나이아신아마이드 EWG 등급")
print(f"임베딩 모델 로드 완료 | 차원: {len(test_vector)}")

# ─────────────────────────────────────────────
# Part 5. FAISS 인덱스 배치 구축
# ─────────────────────────────────────────────
print(f"\nFAISS 인덱스 구축 시작 🚀 (배치 크기: {BATCH_SIZE})")

# 첫 번째 배치로 초기 vectorstore 생성
vectorstore = FAISS.from_documents(
    documents=docs[:BATCH_SIZE],
    embedding=embedding_model,
)
print(f"  배치 1 완료: {min(BATCH_SIZE, len(docs)):,}개")

# 나머지 배치 추가
total_batches = (len(docs) - 1) // BATCH_SIZE + 1
for i in range(1, total_batches):
    start = i * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(docs))
    batch = docs[start:end]

    # 배치별로 임시 vectorstore 만들어서 merge
    batch_vs = FAISS.from_documents(
        documents=batch,
        embedding=embedding_model,
    )
    vectorstore.merge_from(batch_vs)
    print(f"  배치 {i+1}/{total_batches} 완료: {end:,}개 누적")

print(f"\nFAISS 인덱스 구축 완료!")
print(f"저장된 벡터 수: {vectorstore.index.ntotal:,}개")

# 벡터 수 중간 검증
assert vectorstore.index.ntotal == len(docs), \
    f"❌ 벡터 수 불일치! {vectorstore.index.ntotal} ≠ {len(docs)}"
print("✅ 벡터 수 일치 확인")

# ─────────────────────────────────────────────
# Part 6. 기존 폴더 삭제 후 저장
# ─────────────────────────────────────────────
# 기존 폴더가 있으면 삭제 (이전 버전 덮어쓰기 방지)
if os.path.exists(FAISS_SAVE_PATH):
    shutil.rmtree(FAISS_SAVE_PATH)
    print(f"기존 폴더 삭제: {FAISS_SAVE_PATH}")

os.makedirs(FAISS_SAVE_PATH, exist_ok=True)
vectorstore.save_local(FAISS_SAVE_PATH)
print(f"FAISS 인덱스 저장 완료: {FAISS_SAVE_PATH}")

# ─────────────────────────────────────────────
# Part 7. 저장 후 최종 검증
# ─────────────────────────────────────────────
print("\n=== 저장 파일 검증 ===")

faiss_path = os.path.join(FAISS_SAVE_PATH, "index.faiss")
pkl_path   = os.path.join(FAISS_SAVE_PATH, "index.pkl")

faiss_size = os.path.getsize(faiss_path) / 1024 / 1024
pkl_size   = os.path.getsize(pkl_path)   / 1024 / 1024

print(f"index.faiss : {faiss_size:.1f} MB")
print(f"index.pkl   : {pkl_size:.1f} MB")

# 저장된 파일 다시 로드해서 검증
reload_vs = FAISS.load_local(
    FAISS_SAVE_PATH,
    embeddings=embedding_model,
    allow_dangerous_deserialization=True,
)

# docstore 문서 수 확인
docstore_count = len(reload_vs.docstore._dict)
vector_count   = reload_vs.index.ntotal

print(f"\n재로드 후 검증:")
print(f"  벡터 수         : {vector_count:,}개")
print(f"  docstore 문서 수 : {docstore_count:,}개")

if vector_count == docstore_count == len(docs):
    print(f"\n✅ 모든 검증 통과! preset{PRESET_ID} 정상 저장됨")
else:
    print(f"\n❌ 검증 실패!")
    print(f"   원본 청크: {len(docs):,} / 벡터: {vector_count:,} / docstore: {docstore_count:,}")

# 샘플 검색 테스트
print("\n=== 검색 테스트 ===")
results = reload_vs.similarity_search("나이아신아마이드 EWG 등급", k=3)
for i, r in enumerate(results):
    print(f"  [{i+1}] {r.page_content[:80]}")
    print(f"       chunk_type={r.metadata.get('chunk_type')} | coos_score={r.metadata.get('coos_score')} | hw_ewg={r.metadata.get('hw_ewg')}")